## Working directory

In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = "/home/user/NeuroVision-AI-2"

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)
print("Working directory:", os.getcwd())

## Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import pandas as pd
import matplotlib.pyplot as plt

## Configuration and MLflow connection

In [ ]:
from src.config import cfg
from src.mlflow_utils import start_tracking

start_tracking(experiment="neurovision-yolov8")
print(json.dumps(cfg.to_dict(), indent=2)[:600], "...")

## Check the Vast GPU

In [ ]:
from src.train import check_gpu

gpu = check_gpu()
print(json.dumps(gpu, indent=2))

## Augmentation: settings and visual check

In [ ]:
from src.config import AUGMENTATION
from src.train import preview_augmentation

print(json.dumps(AUGMENTATION, indent=2))
preview_augmentation(n=6)       # augmented training images with their boxes

## One-epoch test

In [ ]:
from src.train import train_yolo

smoke = train_yolo(model_name="yolov8n.pt", epochs=1, run_name="smoke-test", mlflow_log=False)
print(smoke["run_dir"], "| duration: %.0f s" % smoke["duration_s"])

## Full training, baseline

In [ ]:
baseline = train_yolo(model_name="yolov8n.pt", run_name="yolov8n-baseline")
print("run dir  :", baseline["run_dir"])
print("duration : %.1f min" % (baseline["duration_s"] / 60))
print(json.dumps(baseline["metrics"], indent=2))

## Read the curves and detect overfitting

In [ ]:
from src.visualization import plot_training_curves

plot_training_curves(baseline["run_dir"])

## Second run, one setting changed

In [ ]:
second = train_yolo(model_name="yolov8s.pt", run_name="yolov8s")
print("duration : %.1f min" % (second["duration_s"] / 60))
print(json.dumps(second["metrics"], indent=2))

## Compare runs and register the champion

In [ ]:
from src.mlflow_utils import compare_runs, register_champion

runs = compare_runs(experiment="neurovision-yolov8")     # DataFrame: params, recall, precision, mAP, duration
display(runs)